In [2]:
import google.protobuf
print(google.protobuf.__version__)

3.19.6


In [3]:
pip install --upgrade protobuf

  Using cached protobuf-6.30.2-cp39-cp39-win_amd64.whl (431 kB)
  Attempting uninstall: protobuf
    Found existing installation: protobuf 3.19.6
    Uninstalling protobuf-3.19.6:
      Successfully uninstalled protobuf-3.19.6
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 5] Access is denied: 'C:\\Users\\pauli\\anaconda3\\Lib\\site-packages\\google\\~%otobuf\\internal\\_api_implementation.cp39-win_amd64.pyd'
Consider using the `--user` option or check the permissions.



In [10]:
pip install tensorflow==2.18.0

  Using cached tensorflow-2.18.0-cp39-cp39-win_amd64.whl (7.5 kB)
  Using cached tensorflow_intel-2.18.0-cp39-cp39-win_amd64.whl (390.0 MB)
  Attempting uninstall: tensorflow-intel
    Found existing installation: tensorflow-intel 2.17.0
    Uninstalling tensorflow-intel-2.17.0:
      Successfully uninstalled tensorflow-intel-2.17.0
  Attempting uninstall: tensorflow
    Found existing installation: tensorflow 2.19.0
    Uninstalling tensorflow-2.19.0:
      Successfully uninstalled tensorflow-2.19.0
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tf-keras 2.19.0 requires tensorflow<2.20,>=2.19, but you have tensorflow 2.18.0 which is incompatible.


In [4]:
pip install --upgrade onnx-tf

Note: you may need to restart the kernel to use updated packages.


In [6]:
import onnx
from onnx_tf.backend import prepare

# Load the ONNX model
onnx_model_path = '../model/yolov8_model.onnx'
onnx_model = onnx.load(onnx_model_path)

# Convert the ONNX model to TensorFlow
tf_rep = prepare(onnx_model)

# Export the TensorFlow model as a SavedModel
saved_model_path = 'yolov8_indiv_teeth_seg_model'
tf_rep.export_graph(saved_model_path)

ModuleNotFoundError: No module named 'tensorflow.python.client'

In [7]:
pip show tensorflow

Name: tensorflow
Version: 2.18.0
Summary: TensorFlow is an open source machine learning framework for everyone.
Home-page: https://www.tensorflow.org/
Author: Google Inc.
Author-email: packages@tensorflow.org
License: Apache 2.0
Location: c:\users\pauli\anaconda3\lib\site-packages
Requires: tensorflow-intel
Required-by: tf_keras
Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install tf2onnx

  Using cached tf2onnx-1.16.1-py3-none-any.whl (455 kB)
Note: you may need to restart the kernel to use updated packages.


In [5]:
import onnx
import onnxruntime as ort
import tensorflow as tf
import numpy as np

# Load ONNX model
onnx_model_path = '../model/yolov8_model.onnx'
onnx_model = onnx.load(onnx_model_path)

# Get model metadata to determine input and output shapes
model_metadata = onnx_model.graph.input[0]
input_shape = [d.dim_value if d.dim_value else None for d in model_metadata.type.tensor_type.shape.dim]
if len(input_shape) == 4:  # [batch, height, width, channels] or [batch, channels, height, width]
    # Adjust shape if needed based on your model's expected input format
    input_shape = [None, None, None, 3]  # Assuming BGR/RGB image input

# Create a wrapper class that uses ONNX Runtime for inference
class ONNXWrapper(tf.keras.Model):
    def __init__(self, onnx_path):
        super(ONNXWrapper, self).__init__()
        self.ort_session = ort.InferenceSession(onnx_path)
        self.input_name = self.ort_session.get_inputs()[0].name
        self.output_names = [output.name for output in self.ort_session.get_outputs()]
    
    @tf.function(input_signature=[tf.TensorSpec(shape=input_shape, dtype=tf.float32)])
    def call(self, inputs):
        # Convert TF tensor to numpy for ONNX Runtime
        inputs_np = inputs.numpy() if isinstance(inputs, tf.Tensor) else inputs
        
        # Run inference with ONNX Runtime
        outputs = self.ort_session.run(self.output_names, {self.input_name: inputs_np})
        
        # Convert output back to TF tensor
        return tf.convert_to_tensor(outputs[0]) if len(outputs) == 1 else [tf.convert_to_tensor(o) for o in outputs]

# Create the model
model = ONNXWrapper(onnx_model_path)

# Save the model in TensorFlow format
saved_model_path = 'yolov8_indiv_teeth_seg_model'
tf.saved_model.save(model, saved_model_path)

AttributeError: module 'tensorflow' has no attribute 'keras'